In [ ]:
# Explore the Ontario National Road Network GeoPackage file.
# Goal: identify which attributes are needed to create a road-network representation for a minimal road network for Ontario and Canada
# This workflow will be generalized to all provinces and territories in the next notebook.

from pathlib import Path
import sqlite3

import pandas as pd
import geopandas as gpd

gpkg_path = Path(
    r"C:\Users\aviga\Research\repos\temoa_geospace\data_files\raw\nrn\ON\NRN_ON_18_0_GPKG_en.gpkg"
)

assert gpkg_path.exists(), f"File not found: {gpkg_path}"

conn = sqlite3.connect(gpkg_path)

In [ ]:
# List all tables in the GPKG file
# We can note that there are tables for junctions, road segments which includes road type and ID

tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
""", conn)

tables

,name
0,NRN_ON_18_0_ADDRANGE
1,NRN_ON_18_0_BLKPASSAGE
2,NRN_ON_18_0_FERRYSEG
3,NRN_ON_18_0_JUNCTION
4,NRN_ON_18_0_ROADSEG
5,NRN_ON_18_0_STRPLANAME
6,NRN_ON_18_0_TOLLPOINT
7,gpkg_contents
8,gpkg_extensions
9,gpkg_geometry_columns


In [3]:
# List all columns in the gpkg_contents table

contents = pd.read_sql_query(
    """
    SELECT *
    FROM gpkg_contents;
    """,
    conn
)

contents

,table_name,data_type,identifier,description,last_change,min_x,min_y,max_x,max_y,srs_id
0,NRN_ON_18_0_STRPLANAME,attributes,NRN_ON_18_0_STRPLANAME,,2025-02-03T14:53:41.198Z,NaN,NaN,NaN,NaN,0
1,NRN_ON_18_0_TOLLPOINT,features,NRN_ON_18_0_TOLLPOINT,,2025-02-03T14:53:41.211Z,-83.036431,41.763366,-75.980277,44.368237,4617
2,NRN_ON_18_0_FERRYSEG,features,NRN_ON_18_0_FERRYSEG,,2025-02-03T14:53:41.221Z,-93.827784,41.676546,-74.882287,51.279179,4617
3,NRN_ON_18_0_JUNCTION,features,NRN_ON_18_0_JUNCTION,,2025-02-03T14:53:43.762Z,-95.153425,41.676546,-74.343857,56.082803,4617
4,NRN_ON_18_0_ADDRANGE,attributes,NRN_ON_18_0_ADDRANGE,,2025-02-03T14:53:43.766Z,NaN,NaN,NaN,NaN,0
5,NRN_ON_18_0_ROADSEG,features,NRN_ON_18_0_ROADSEG,,2025-02-03T14:53:47.051Z,-95.153425,41.734662,-74.343857,56.082803,4617
6,NRN_ON_18_0_BLKPASSAGE,features,NRN_ON_18_0_BLKPASSAGE,,2025-02-03T14:53:47.076Z,-95.091639,41.931241,-74.363732,52.219129,4617


In [9]:
# Define a variable for the road segments table name which will later be used to read the road segments data into a GeoDataFrame
road_table = "NRN_ON_18_0_ROADSEG"


In [ ]:
# Count the total number of road segments in the road segments table

total_segments = pd.read_sql_query(
    f"""
    SELECT COUNT(*) AS total_segments
    FROM {road_table};
    """,
    conn
)

total_segments

,total_segments
0,651662


In [12]:
# Determine the types of roads used to classify the road segments and how many segments are in each class
# Based on the ROADCLASS distribution, the classes Freeway, Expressway / Highway,
# Arterial, and Ramp form a plausible minimum viable road-network representation for Ontario freight-access modelling.

roadclass_counts = pd.read_sql_query(
    f"""
    SELECT ROADCLASS,
           COUNT(*) AS segments
    FROM {road_table}
    GROUP BY ROADCLASS
    ORDER BY segments DESC;
    """,
    conn
)

roadclass_counts

# Close SQLite connection
conn.close()

In [ ]:
# The strict highway backbone contains Freeway, Expressway / Highway, and Ramp.
# Adding Arterial roads creates a broader freight-access network that may better
# Connect industrial sites, ports, terminals, and TEMOA region centroids.
# We might consider two possible road network representations for Ontario: 
# A strict highway backbone and a broader freight-access network.

highway_backbone_classes = [
    "Freeway",
    "Expressway / Highway",
    "Ramp",
]

freight_access_classes = [
    "Freeway",
    "Expressway / Highway",
    "Ramp",
    "Arterial",
]